In [ ]:
# config bootstrap (auto-added): resolve repo paths from config.py
import os as _os, sys as _sys
_h = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_h, 'config.py')) and _os.path.dirname(_h) != _h:
    _h = _os.path.dirname(_h)
_sys.path.insert(0, _h)
import config as _cfg

# Cross-Modal Consistency Module — BLIP ITM

This notebook loads samples from the NewsClipPings dataset, uses BLIP's dedicated
**Image-Text Matching (ITM) head** to compute match probabilities, and evaluates
whether these scores can distinguish real (matched) vs. falsified (mismatched)
image-caption pairs.

In [1]:
import json
import os

import torch
from PIL import Image
from transformers import BlipProcessor, BlipForImageTextRetrieval
import matplotlib.pyplot as plt
import pandas as pd

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

d:\Pics Can Lie\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.6.0+cu124
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 8.0 GB


In [2]:
# ── Config ──
DATASET_ROOT    = _os.path.join(str(_cfg.ROOT), 'datasets', 'dataset')
ANNOTATIONS_PATH = os.path.join(DATASET_ROOT, "data", "NewsClipPings", "merged_balanced", "train.json")
METADATA_PATH    = os.path.join(DATASET_ROOT, "data", "NewsClipPings", "metadata", "train.json")

# Image paths in metadata use 'visual_news/origin/...' but on disk they are at 'dataset/origin/origin/...'
IMAGE_BASE = os.path.join(DATASET_ROOT, "origin")  # visual_news/origin/X -> dataset/origin/origin/X

def resolve_image_path(meta_image_path: str) -> str:
    """Convert metadata image_path to actual disk path."""
    # meta_image_path = 'visual_news/origin/guardian/images/0540/632.jpg'
    # strip 'visual_news/' prefix -> 'origin/guardian/images/0540/632.jpg'
    rel = meta_image_path.replace("visual_news/", "", 1)
    return os.path.join(IMAGE_BASE, rel)

print("Paths configured.")

Paths configured.


In [3]:
# ── Load annotations & metadata ──
with open(ANNOTATIONS_PATH, "r", encoding="utf-8") as f:
    train_data = json.load(f)
annotations = train_data["annotations"]

with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print(f"Annotations: {len(annotations):,}")
print(f"Metadata entries: {len(metadata):,}")

# ── Sample 100 real + 100 falsified pairs ──
NUM_PER_CLASS = 100
real_samples = []
fake_samples = []

for ann in annotations:
    if len(real_samples) >= NUM_PER_CLASS and len(fake_samples) >= NUM_PER_CLASS:
        break

    art_id = str(ann["id"])
    img_id = str(ann["image_id"])

    if art_id not in metadata or img_id not in metadata:
        continue

    img_path = resolve_image_path(metadata[img_id]["image_path"])
    if not os.path.isfile(img_path):
        continue

    caption = metadata[art_id]["caption"]

    entry = {
        "article_id": ann["id"],
        "image_id": ann["image_id"],
        "caption": caption,
        "image_path": img_path,
        "falsified": ann["falsified"],
        "similarity_score": ann["similarity_score"],
        "source": metadata[art_id]["source"],
    }

    if not ann["falsified"] and len(real_samples) < NUM_PER_CLASS:
        real_samples.append(entry)
    elif ann["falsified"] and len(fake_samples) < NUM_PER_CLASS:
        fake_samples.append(entry)

samples = real_samples + fake_samples
print(f"\nSelected {len(real_samples)} real + {len(fake_samples)} falsified = {len(samples)} samples")

Annotations: 71,072
Metadata entries: 385,003

Selected 100 real + 100 falsified = 200 samples


In [4]:
# ── Load BLIP ITM model ──
# This model has a dedicated Image-Text Matching head that outputs
# a probability of whether an image and text are a genuine pair.
MODEL_NAME = "Salesforce/blip-itm-base-coco"

print(f"Loading {MODEL_NAME}...")
processor = BlipProcessor.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BlipForImageTextRetrieval.from_pretrained(MODEL_NAME).to(device)
model.eval()

print(f"Model loaded on: {device}")
param_count = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Parameters: {param_count:.1f}M")

Loading Salesforce/blip-itm-base-coco...


The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|██████████| 472/472 [00:00<00:00, 18874.35it/s]
BlipForImageTextRetrieval LOAD REPORT from: Salesforce/blip-itm-base-coco
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_encoder.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded on: cuda
Parameters: 223.7M


In [5]:
# ── Compute ITM scores (200 samples) ──
from tqdm import tqdm

results = []

for i, s in enumerate(tqdm(samples, desc="Computing ITM scores")):
    image = Image.open(s["image_path"]).convert("RGB")

    inputs = processor(images=image, text=s["caption"], return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs, use_itm_head=True)
        itm_probs = torch.softmax(outputs.itm_score, dim=1)
        match_prob = itm_probs[0, 1].item()

    results.append({
        "article_id": s["article_id"],
        "image_id": s["image_id"],
        "ground_truth": "REAL" if not s["falsified"] else "FAKE",
        "caption": s["caption"][:60] + "...",
        "itm_score": match_prob,
        "source": s["source"],
    })

print(f"\nDone! Processed {len(results)} samples.")

Computing ITM scores: 100%|██████████| 200/200 [00:12<00:00, 15.72it/s]


Done! Processed 200 samples.


In [6]:
# ── Evaluation: Accuracy, Precision, Recall, F1 ──
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

df = pd.DataFrame(results)

# Predict REAL if ITM score > 0.5, else FAKE
df["prediction"] = df["itm_score"].apply(lambda x: "REAL" if x > 0.5 else "FAKE")

y_true = (df["ground_truth"] == "FAKE").astype(int)  # 1 = FAKE, 0 = REAL
y_pred = (df["prediction"] == "FAKE").astype(int)

acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec  = recall_score(y_true, y_pred)
f1   = f1_score(y_true, y_pred)

real_scores = df[df["ground_truth"] == "REAL"]["itm_score"]
fake_scores = df[df["ground_truth"] == "FAKE"]["itm_score"]

print("=" * 60)
print("BLIP ITM — 200 SAMPLE EVALUATION (100 real, 100 fake)")
print("=" * 60)

print(f"\n  Accuracy  : {acc:.2%}")
print(f"  Precision : {prec:.2%}  (of predicted FAKE, how many are truly fake)")
print(f"  Recall    : {rec:.2%}  (of actual FAKE, how many were caught)")
print(f"  F1 Score  : {f1:.2%}")

print(f"\n{'─' * 60}")
print(f"  REAL ITM scores — mean: {real_scores.mean():.4f}  std: {real_scores.std():.4f}")
print(f"  FAKE ITM scores — mean: {fake_scores.mean():.4f}  std: {fake_scores.std():.4f}")
print(f"  Separation      : {real_scores.mean() - fake_scores.mean():.4f}")

print(f"\n{'─' * 60}")
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=["REAL", "FAKE"]))

print("Confusion Matrix:")
cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm, index=["Actual REAL", "Actual FAKE"], columns=["Pred REAL", "Pred FAKE"])
display(cm_df)

BLIP ITM — 200 SAMPLE EVALUATION (100 real, 100 fake)

  Accuracy  : 72.50%
  Precision : 69.57%  (of predicted FAKE, how many are truly fake)
  Recall    : 80.00%  (of actual FAKE, how many were caught)
  F1 Score  : 74.42%

────────────────────────────────────────────────────────────
  REAL ITM scores — mean: 0.6373  std: 0.3765
  FAKE ITM scores — mean: 0.2143  std: 0.3486
  Separation      : 0.4230

────────────────────────────────────────────────────────────
Classification Report:
              precision    recall  f1-score   support

        REAL       0.76      0.65      0.70       100
        FAKE       0.70      0.80      0.74       100

    accuracy                           0.72       200
   macro avg       0.73      0.73      0.72       200
weighted avg       0.73      0.72      0.72       200

Confusion Matrix:


,Pred REAL,Pred FAKE
Actual REAL,65,35
Actual FAKE,20,80


In [ ]:
# ── ITM Score Distribution ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax = axes[0]
ax.hist(real_scores, bins=20, alpha=0.7, color="#2ecc71", label="REAL", edgecolor="white")
ax.hist(fake_scores, bins=20, alpha=0.7, color="#e74c3c", label="FAKE", edgecolor="white")
ax.axvline(x=0.5, color="orange", linestyle="--", linewidth=2, label="Threshold (0.5)")
ax.set_xlabel("ITM Match Probability")
ax.set_ylabel("Count")
ax.set_title("Distribution of ITM Scores")
ax.legend()

# Box plot
ax = axes[1]
data = [real_scores.values, fake_scores.values]
bp = ax.boxplot(data, labels=["REAL", "FAKE"], patch_artist=True)
bp["boxes"][0].set_facecolor("#2ecc71")
bp["boxes"][1].set_facecolor("#e74c3c")
ax.axhline(y=0.5, color="orange", linestyle="--", linewidth=2, label="Threshold (0.5)")
ax.set_ylabel("ITM Match Probability")
ax.set_title("ITM Score Distribution by Class")
ax.legend()

plt.suptitle(f"BLIP ITM — 200 Samples | Accuracy: {acc:.1%} | F1: {f1:.1%}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("blip_itm_200_results.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Misclassified examples ──
df["correct"] = df["ground_truth"] == df["prediction"]
wrong = df[~df["correct"]]

print(f"Misclassified: {len(wrong)}/{len(df)}")
print()
if len(wrong) > 0:
    print("Sample misclassifications:")
    for _, row in wrong.head(10).iterrows():
        print(f"  [{row['ground_truth']}→{row['prediction']}] ITM={row['itm_score']:.4f} | {row['caption']}")